# CodeQA Mini Attention — Perbandingan Terkontrol GRU vs GRU+Attention

**Konteks Proyek:**
- Paper Acuan: *CodeQA: A Question Answering Dataset for Source Code Comprehension* (Liu & Wan, EMNLP 2021).
- Tema Intern: **Reproduce → Rethink → Reinvent**.
- **Tujuan Eksperimen:** Mengevaluasi secara empiris hipotesis *information bottleneck* pada model sekuensial kecil melalui perbandingan terkontrol (*fair comparison*) antara:
  1. **Varian A (Baseline Kontrol):** Bidirectional GRU Encoder + Unidirectional GRU Decoder tanpa attention.
  2. **Varian B (Intervensi/Metode Baru):** Bidirectional GRU Encoder + Unidirectional GRU Decoder dengan **Luong General Attention**.
- **Lingkungan & Batasan Hardware:**
  - NVIDIA GeForce RTX 3050 Laptop GPU (VRAM 4 GB).
  - Target model: Model kompak (< 5 juta parameter) yang dilatih dari nol (*from scratch*).
  - Data: 10.000 train, 500 dev, dan 500 test dari split resmi Python CodeQA.
  - Test set **dikarantina penuh** selama perancangan vocab, tuning, dan training.

In [1]:
import os, sys, time, json, math, random, copy, re
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("mini_attention_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device Aktif: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"Output Directory: {OUTPUT_DIR.resolve()}")

Device Aktif: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM: 3.73 GB
Output Directory: /home/inaka/Codes/LAB_IC/CodeQA/mini_attention_results


## 1. Pemuatan Data & Karantina Test Set

Kami memuat subset resmi Python:
- **10.000 sampel Train** dari `data/python/train/`
- **500 sampel Dev** dari `data/python/dev/`
- **500 sampel Test** dari `data/python/test/`

> **Protokol Integritas Ilmiah:** Test set hanya dimuat ke memori untuk verifikasi alignment dan deteksi overlap kode. Test set **dikarantina total** dan dilarang digunakan dalam pembangunan kosakata, tuning hyperparameter, maupun pemilihan checkpoint.

In [2]:
def load_split_rows(data_dir, split_name, limit):
    q_file = Path(data_dir) / split_name / f"{split_name}.question"
    c_file = Path(data_dir) / split_name / f"{split_name}.code"
    a_file = Path(data_dir) / split_name / f"{split_name}.answer"
    
    rows = []
    with open(q_file, "r", encoding="utf-8") as fq, \
         open(c_file, "r", encoding="utf-8") as fc, \
         open(a_file, "r", encoding="utf-8") as fa:
        for idx, (q, c, a) in enumerate(zip(fq, fc, fa)):
            if idx >= limit:
                break
            rows.append({
                "id": idx,
                "question": q.strip(),
                "code": c.strip(),
                "answer": a.strip()
            })
    return rows

raw_data = {
    "train": load_split_rows("data/python", "train", 10000),
    "dev": load_split_rows("data/python", "dev", 500),
    "test": load_split_rows("data/python", "test", 500)
}

assert len(raw_data["train"]) == 10000
assert len(raw_data["dev"]) == 500
assert len(raw_data["test"]) == 500

# Deteksi overlap string kode antara train subset dan test subset
train_codes = set(r["code"] for r in raw_data["train"])
dev_codes = set(r["code"] for r in raw_data["dev"])
test_codes = set(r["code"] for r in raw_data["test"])

overlap_train_test = train_codes.intersection(test_codes)
overlap_train_dev = train_codes.intersection(dev_codes)

print("Jumlah Data Terpilih:")
print(f"  - Train: {len(raw_data['train'])} baris")
print(f"  - Dev  : {len(raw_data['dev'])} baris")
print(f"  - Test : {len(raw_data['test'])} baris (DIKARANTINA)")
print(f"Overlap kode Train & Dev : {len(overlap_train_dev)} cuplikan")
print(f"Overlap kode Train & Test: {len(overlap_train_test)} cuplikan (dilaporkan transparan, tidak dipindahkan)")

Jumlah Data Terpilih:
  - Train: 10000 baris
  - Dev  : 500 baris
  - Test : 500 baris (DIKARANTINA)
Overlap kode Train & Dev : 1 cuplikan
Overlap kode Train & Test: 1 cuplikan (dilaporkan transparan, tidak dipindahkan)


## 2. Tokenisasi & Pembangunan Kosakata (Murni dari Train)

- **Tokenizer:** Menggunakan ekspresi reguler kata dan tanda baca: `re.findall(r"\w+|[^\w\s]", text.lower())`.
- **Special Tokens:**
  - `PAD = 0` (`<pad>`)
  - `BOS = 1` (`<bos>`)
  - `EOS = 2` (`<eos>`)
  - `UNK = 3` (`<unk>`)
  - `SEP = 4` (`<sep>`)
- **Kosakata:** Dibangun murni dari 10.000 data train dengan batas kandidat 8.000 token.
- **Evaluasi OOV:** Rasio kata di luar kosakata (*Out-of-Vocabulary*) diukur secara murni pada **Dev Set**.

In [3]:
PAD, BOS, EOS, UNK, SEP = range(5)
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "<sep>"]

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

# Hitung frekuensi token HANYA dari subset train
train_token_counts = Counter()
for row in raw_data["train"]:
    train_token_counts.update(tokenize(row["question"]))
    train_token_counts.update(tokenize(row["code"]))
    train_token_counts.update(tokenize(row["answer"]))

VOCAB_SIZE = 8000
vocab = SPECIAL_TOKENS + [w for w, _ in train_token_counts.most_common(VOCAB_SIZE - len(SPECIAL_TOKENS)) if w not in SPECIAL_TOKENS]
stoi = {w: i for i, w in enumerate(vocab)}

print(f"Total token unik di Train 10k: {len(train_token_counts):,}")
print(f"Ukuran Kosakata Terpilih    : {len(vocab):,}")

# Evaluasi OOV pada 500 Dev (BUKAN Test)
dev_answer_tokens = [tok for row in raw_data["dev"] for tok in tokenize(row["answer"])]
dev_input_tokens = [tok for row in raw_data["dev"] for tok in tokenize(row["question"]) + tokenize(row["code"])]

dev_ans_oov = sum(1 for t in dev_answer_tokens if t not in stoi)
dev_inp_oov = sum(1 for t in dev_input_tokens if t not in stoi)

print(f"OOV pada Dev Answers: {dev_ans_oov / len(dev_answer_tokens) * 100:.2f}% ({dev_ans_oov}/{len(dev_answer_tokens)})")
print(f"OOV pada Dev Inputs : {dev_inp_oov / len(dev_input_tokens) * 100:.2f}% ({dev_inp_oov}/{len(dev_input_tokens)})")

Total token unik di Train 10k: 17,454
Ukuran Kosakata Terpilih    : 8,000
OOV pada Dev Answers: 8.31% (192/2310)
OOV pada Dev Inputs : 7.57% (2576/34019)


## 3. Prapemrosesan Tensor & Analisis Pemotongan (*Truncation*)

Berdasarkan analisis statistik persentil data:
- `MAX_SOURCE = 160` (mencakup >95% data input tanpa pemotongan)
- `MAX_TARGET = 24` (mencakup >99% target jawaban tanpa pemotongan)

In [4]:
MAX_SOURCE = 160
MAX_TARGET = 24

def build_tensor_dataset(rows):
    sources, lengths, targets = [], [], []
    truncated_src = 0
    truncated_tgt = 0
    
    for r in rows:
        q_toks = [stoi.get(t, UNK) for t in tokenize(r["question"])]
        c_toks = [stoi.get(t, UNK) for t in tokenize(r["code"])]
        a_toks = [stoi.get(t, UNK) for t in tokenize(r["answer"])]
        
        full_src = q_toks + [SEP] + c_toks
        if len(full_src) > MAX_SOURCE:
            truncated_src += 1
        src = full_src[:MAX_SOURCE]
        
        full_tgt = a_toks
        if len(full_tgt) > MAX_TARGET - 2:
            truncated_tgt += 1
        tgt = [BOS] + full_tgt[:MAX_TARGET - 2] + [EOS]
        
        lengths.append(len(src))
        sources.append(src + [PAD] * (MAX_SOURCE - len(src)))
        targets.append(tgt + [PAD] * (MAX_TARGET - len(tgt)))
        
    ds = TensorDataset(torch.tensor(sources, dtype=torch.long),
                       torch.tensor(lengths, dtype=torch.long),
                       torch.tensor(targets, dtype=torch.long))
    trunc_info = {
        "src_trunc_pct": (truncated_src / len(rows)) * 100,
        "tgt_trunc_pct": (truncated_tgt / len(rows)) * 100
    }
    return ds, trunc_info

train_dataset, train_trunc = build_tensor_dataset(raw_data["train"])
dev_dataset, dev_trunc = build_tensor_dataset(raw_data["dev"])
test_dataset, test_trunc = build_tensor_dataset(raw_data["test"])

print(f"Train Dataset Tensor: {tuple(train_dataset.tensors[0].shape)}")
print(f"  - Input terpotong (>160): {train_trunc['src_trunc_pct']:.2f}%")
print(f"  - Target terpotong (>24): {train_trunc['tgt_trunc_pct']:.2f}%")
print(f"Dev Dataset Tensor  : {tuple(dev_dataset.tensors[0].shape)}")

Train Dataset Tensor: (10000, 160)
  - Input terpotong (>160): 4.40%
  - Target terpotong (>24): 0.15%
Dev Dataset Tensor  : (500, 160)


## 4. Arsitektur Model: Varian A (Kontrol) vs Varian B (Luong Attention)

Untuk menjamin perbandingan yang adil (*fair comparison*):
- **Shared Encoder:** 1-layer Bidirectional GRU (`hidden_size // 2 = 128` per arah $\to$ total 256).
- **Shared Initial State Decoder:** $h_{\text{init}} = [h_{\text{fwd}}; h_{\text{bwd}}]$ (dimensi 256).
- **Shared Decoder GRU:** 1-layer Unidirectional GRU (dimensi 256).
- **Shared Embedding:** Embedding layer bersama (dimensi 128) antara encoder dan decoder.
- **Pembeda Tunggal:**
  - **Varian A:** Output decoder GRU langsung diproyeksikan ke kosakata lewat `Linear(256, 8000)`.
  - **Varian B:** Output decoder GRU menghitung bobot Luong General Attention ($\text{softmax}(s_t^T W_a h_i)$) dengan padding mask, membentuk context vector $c_t$, digabungkan menjadi attentional vector $\tilde{s}_t = \tanh(W_c [c_t; s_t])$, lalu diproyeksikan ke kosakata lewat `Linear(256, 8000)`.

In [5]:
EMBED_DIM = 128
HIDDEN_DIM = 256

class SharedEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, embedding):
        super().__init__()
        self.embedding = embedding
        self.gru = nn.GRU(embed_dim, hidden_dim // 2, batch_first=True, bidirectional=True)
        
    def forward(self, source, lengths):
        embeds = self.embedding(source)
        packed = nn.utils.rnn.pack_padded_sequence(embeds, lengths.cpu(), batch_first=True, enforce_sorted=False)
        enc_out, hidden = self.gru(packed)
        enc_out, _ = nn.utils.rnn.pad_packed_sequence(enc_out, batch_first=True, total_length=source.size(1))
        # Gabungkan arah forward (hidden[0]) dan backward (hidden[1])
        dec_init = torch.cat([hidden[0], hidden[1]], dim=-1).unsqueeze(0) # (1, B, hidden_dim)
        return enc_out, dec_init

class DecoderPlain(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, embedding):
        super().__init__()
        self.embedding = embedding
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
    def step(self, token, hidden, enc_out, mask):
        embed = self.embedding(token)
        output, hidden = self.gru(embed, hidden)
        logits = self.fc_out(output.squeeze(1))
        return logits, hidden, None

class DecoderAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, embedding):
        super().__init__()
        self.embedding = embedding
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.W_a = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_c = nn.Linear(2 * hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
    def step(self, token, hidden, enc_out, mask):
        embed = self.embedding(token)
        output, hidden = self.gru(embed, hidden) # (B, 1, H)
        
        # Luong General Attention: score = s_t^T * W_a * h_i
        proj_enc = self.W_a(enc_out) # (B, T_x, H)
        scores = torch.bmm(output, proj_enc.transpose(1, 2)) # (B, 1, T_x)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(1), -1e9)
        attn_weights = F.softmax(scores, dim=-1) # (B, 1, T_x)
        context = torch.bmm(attn_weights, enc_out) # (B, 1, H)
        
        concat = torch.cat([context, output], dim=-1) # (B, 1, 2H)
        combined = torch.tanh(self.W_c(concat)) # (B, 1, H)
        logits = self.fc_out(combined.squeeze(1)) # (B, V)
        return logits, hidden, attn_weights

class Seq2SeqModel(nn.Module):
    def __init__(self, encoder, decoder, pad_idx=PAD, bos_idx=BOS, eos_idx=EOS, sep_idx=SEP):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.sep_idx = sep_idx
        
    def forward(self, source, lengths, target, teacher_forcing_ratio=0.5):
        B, max_len = target.shape
        enc_out, dec_hidden = self.encoder(source, lengths)
        mask = (source != self.pad_idx)
        
        logits_list = []
        token = target[:, 0:1] # BOS
        for t in range(1, max_len):
            logits, dec_hidden, _ = self.decoder.step(token, dec_hidden, enc_out, mask)
            logits_list.append(logits.unsqueeze(1))
            use_tf = (torch.rand(1).item() < teacher_forcing_ratio) if self.training else False
            token = target[:, t:t+1] if use_tf else logits.argmax(-1, keepdim=True)
        return torch.cat(logits_list, dim=1)
        
    @torch.no_grad()
    def generate(self, source, lengths, max_target=MAX_TARGET):
        self.eval()
        B = source.size(0)
        enc_out, dec_hidden = self.encoder(source, lengths)
        mask = (source != self.pad_idx)
        
        token = torch.full((B, 1), self.bos_idx, dtype=torch.long, device=source.device)
        done = torch.zeros(B, dtype=torch.bool, device=source.device)
        generated = []
        for _ in range(max_target - 1):
            logits, dec_hidden, _ = self.decoder.step(token, dec_hidden, enc_out, mask)
            logits[:, [self.pad_idx, self.bos_idx, self.sep_idx]] = -float("inf")
            token = logits.argmax(-1, keepdim=True)
            token = torch.where(done[:, None], torch.full_like(token, self.eos_idx), token)
            generated.append(token)
            done |= token.squeeze(1).eq(self.eos_idx)
            if done.all():
                break
        return torch.cat(generated, dim=1) if generated else torch.full((B, 1), self.eos_idx, device=source.device)

def build_model_a():
    shared_embed = nn.Embedding(len(vocab), EMBED_DIM, padding_idx=PAD)
    enc = SharedEncoder(len(vocab), EMBED_DIM, HIDDEN_DIM, shared_embed)
    dec = DecoderPlain(len(vocab), EMBED_DIM, HIDDEN_DIM, shared_embed)
    return Seq2SeqModel(enc, dec)

def build_model_b():
    shared_embed = nn.Embedding(len(vocab), EMBED_DIM, padding_idx=PAD)
    enc = SharedEncoder(len(vocab), EMBED_DIM, HIDDEN_DIM, shared_embed)
    dec = DecoderAttention(len(vocab), EMBED_DIM, HIDDEN_DIM, shared_embed)
    return Seq2SeqModel(enc, dec)

model_a = build_model_a().to(DEVICE)
model_b = build_model_b().to(DEVICE)

params_a = sum(p.numel() for p in model_a.parameters())
params_b = sum(p.numel() for p in model_b.parameters())

print(f"Total Parameter Varian A (GRU Polos)    : {params_a:,}")
print(f"Total Parameter Varian B (GRU+Attention): {params_b:,}")
assert params_a < 5_000_000 and params_b < 5_000_000

Total Parameter Varian A (GRU Polos)    : 3,574,592
Total Parameter Varian B (GRU+Attention): 3,771,456


## 5. Uji Sanity Check Mekanis (Probe 4 Sampel)

Uji ini berfungsi menguji **integritas mekanis kode** sebelum training skala penuh:
1. Memastikan forward pass menghasilkan dimensi tensor logits yang presisi.
2. Memastikan gradien terhitung dan bernilai *finite* (`torch.isfinite`).
3. Memastikan bobot atensi menjumlahkan ke 1,0 dan token `<pad>` menerima bobot mendekati nol.
4. Memastikan loss menunjukkan penurunan setelah beberapa langkah optimasi.

In [6]:
probe_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
probe_src, probe_len, probe_tgt = next(iter(probe_loader))
probe_src, probe_len, probe_tgt = probe_src.to(DEVICE), probe_len.to(DEVICE), probe_tgt.to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)

# 1. Forward pass check
out_a = model_a(probe_src, probe_len, probe_tgt)
out_b = model_b(probe_src, probe_len, probe_tgt)
assert out_a.shape == (4, MAX_TARGET - 1, len(vocab))
assert out_b.shape == (4, MAX_TARGET - 1, len(vocab))

# 2. Gradient finiteness check
loss_b = criterion(out_b.reshape(-1, len(vocab)), probe_tgt[:, 1:].reshape(-1))
loss_b.backward()
for name, param in model_b.named_parameters():
    if param.grad is not None:
        assert torch.isfinite(param.grad).all(), f"Gradien non-finite pada {name}"
model_b.zero_grad(set_to_none=True)

# 3. Attention sum and pad masking check
with torch.no_grad():
    enc_out, dec_hid = model_b.encoder(probe_src, probe_len)
    mask = (probe_src != PAD)
    tok = probe_tgt[:, 0:1]
    _, _, attn_w = model_b.decoder.step(tok, dec_hid, enc_out, mask)
    attn_sum = attn_w.sum(dim=-1)
    assert torch.allclose(attn_sum, torch.ones_like(attn_sum), atol=1e-4), "Softmax atensi tidak berjumlah 1.0"
    pad_positions = ~mask
    if pad_positions.any():
        assert (attn_w.squeeze(1)[pad_positions] < 1e-4).all(), "Token padding masih menerima atensi signifikan"

# 4. Probe optimization trend check (10 langkah)
opt_probe = torch.optim.Adam(model_b.parameters(), lr=0.01)
initial_loss = float(loss_b)
for _ in range(10):
    opt_probe.zero_grad()
    o = model_b(probe_src, probe_len, probe_tgt, teacher_forcing_ratio=1.0)
    l = criterion(o.reshape(-1, len(vocab)), probe_tgt[:, 1:].reshape(-1))
    l.backward()
    opt_probe.step()
final_loss = float(l)
assert final_loss < initial_loss, f"Loss probe gagal turun: {initial_loss:.4f} -> {final_loss:.4f}"

# Re-inisialisasi model bersih setelah uji probe
model_a = build_model_a().to(DEVICE)
model_b = build_model_b().to(DEVICE)

print("SANITY CHECK MEKANIS LULUS:")
print("  - Shape forward valid")
print("  - Gradien terverifikasi finite")
print("  - Bobot atensi menjumlahkan ke 1.0 dan masking padding aktif")
print(f"  - Tren penurunan loss probe terverifikasi: {initial_loss:.4f} -> {final_loss:.4f}")

/tmp/ipykernel_846254/3536894661.py:35: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  initial_loss = float(loss_b)


SANITY CHECK MEKANIS LULUS:
  - Shape forward valid
  - Gradien terverifikasi finite
  - Bobot atensi menjumlahkan ke 1.0 dan masking padding aktif
  - Tren penurunan loss probe terverifikasi: 8.9573 -> 0.1547


## 6. Benchmark Waktu & Memori GPU (100 Minibatch)

Benchmark empiris 100 minibatch (10 warmup + 90 measured) pada batch size 32 untuk mengukur peak VRAM dan throughput aktual.

In [7]:
BATCH_SIZE = 32

def run_hardware_benchmark(name, model):
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss(ignore_index=PAD)
    
    # 10 batch warmup
    for _ in range(10):
        src = torch.randint(1, len(vocab), (BATCH_SIZE, MAX_SOURCE), device=DEVICE)
        l = torch.randint(20, MAX_SOURCE, (BATCH_SIZE,), device=DEVICE)
        tgt = torch.randint(1, len(vocab), (BATCH_SIZE, MAX_TARGET), device=DEVICE)
        opt.zero_grad()
        out = model(src, l, tgt, teacher_forcing_ratio=0.5)
        loss = crit(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    # 90 batch measured
    for _ in range(90):
        src = torch.randint(1, len(vocab), (BATCH_SIZE, MAX_SOURCE), device=DEVICE)
        l = torch.randint(20, MAX_SOURCE, (BATCH_SIZE,), device=DEVICE)
        tgt = torch.randint(1, len(vocab), (BATCH_SIZE, MAX_TARGET), device=DEVICE)
        opt.zero_grad()
        out = model(src, l, tgt, teacher_forcing_ratio=0.5)
        loss = crit(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        peak_vram_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    else:
        peak_vram_mb = 0.0
        
    elapsed = time.perf_counter() - t0
    step_ms = (elapsed / 90) * 1000
    throughput = (90 * BATCH_SIZE) / elapsed
    epoch_batches = math.ceil(len(raw_data["train"]) / BATCH_SIZE)
    est_train_epoch_sec = (epoch_batches * elapsed) / 90
    
    return {
        "variant": name,
        "step_ms": step_ms,
        "throughput_samples_sec": throughput,
        "est_train_epoch_sec": est_train_epoch_sec,
        "peak_vram_mb": peak_vram_mb
    }

bench_a = run_hardware_benchmark("Varian A (Plain GRU)", model_a)
bench_b = run_hardware_benchmark("Varian B (GRU+Attention)", model_b)

# Re-inisialisasi model bersih setelah benchmark
model_a = build_model_a().to(DEVICE)
model_b = build_model_b().to(DEVICE)

# Simpan laporan benchmark
benchmark_report = {"variant_a": bench_a, "variant_b": bench_b}
(OUTPUT_DIR / "benchmark_report.json").write_text(json.dumps(benchmark_report, indent=2))

print("=== HASIL BENCHMARK EMPIRIS (BATCH 32) ===")
print(f"Varian A: Step {bench_a['step_ms']:.1f} ms | Est. Train Epoch {bench_a['est_train_epoch_sec']:.1f} s | Peak VRAM {bench_a['peak_vram_mb']:.1f} MB")
print(f"Varian B: Step {bench_b['step_ms']:.1f} ms | Est. Train Epoch {bench_b['est_train_epoch_sec']:.1f} s | Peak VRAM {bench_b['peak_vram_mb']:.1f} MB")

# Penentuan epoch dinamis agar total durasi A + B aman (< 30 menit)
MAX_EPOCHS = 10
total_est_seconds = (bench_a['est_train_epoch_sec'] + bench_b['est_train_epoch_sec'] + 5) * MAX_EPOCHS
print(f"\nAlokasi Epoch: {MAX_EPOCHS} epoch per varian (Proyeksi total training: ~{total_est_seconds / 60:.1f} menit, sangat aman di bawah batas 2 jam)")

=== HASIL BENCHMARK EMPIRIS (BATCH 32) ===
Varian A: Step 25.7 ms | Est. Train Epoch 8.0 s | Peak VRAM 280.0 MB
Varian B: Step 60.6 ms | Est. Train Epoch 19.0 s | Peak VRAM 418.4 MB

Alokasi Epoch: 10 epoch per varian (Proyeksi total training: ~5.3 menit, sangat aman di bawah batas 2 jam)


## 7. Training & Early Stopping Berbasis Dev F1

- **Pencegahan Overfitting & Pemilihan Checkpoint:** Di setiap akhir epoch, model dievaluasi pada 500 Dev set (Dev Loss dan Dev Generation Token F1).
- Checkpoint terbaik disimpan berdasarkan skor **Dev Token F1** tertinggi (Dev loss sebagai pemecah seri).
- Test set tetap berada dalam karantina total selama proses ini berlangsung.

In [8]:
def decode_tokens(ids):
    tokens = []
    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, BOS, SEP):
            tokens.append(vocab[idx])
    return " ".join(tokens)

def compute_metrics(predictions, references):
    em_count = 0
    f1_list = []
    yes_no_em = 0
    non_yes_no_em = 0
    
    for pred, ref in zip(predictions, references):
        p_toks = tokenize(pred)
        r_toks = tokenize(ref)
        is_em = (p_toks == r_toks)
        if is_em:
            em_count += 1
            if ref.strip().lower() in ("yes", "no"):
                yes_no_em += 1
            else:
                non_yes_no_em += 1
        overlap = sum((Counter(p_toks) & Counter(r_toks)).values())
        if not p_toks and not r_toks:
            f1 = 1.0
        elif not p_toks or not r_toks:
            f1 = 0.0
        else:
            prec = overlap / len(p_toks)
            rec = overlap / len(r_toks)
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1_list.append(f1)
        
    pred_counts = Counter(predictions)
    top_pred, top_count = pred_counts.most_common(1)[0] if pred_counts else ("", 0)
    
    return {
        "EM": (em_count / len(predictions)) * 100,
        "token_F1": (sum(f1_list) / len(f1_list)) * 100,
        "unique_predictions": len(pred_counts),
        "top_1_prediction": top_pred,
        "top_1_concentration": (top_count / len(predictions)) * 100,
        "total_em_count": em_count,
        "yes_no_em_count": yes_no_em,
        "non_yes_no_em_count": non_yes_no_em
    }

@torch.no_grad()
def evaluate_dev(model, dataset, raw_split_rows):
    model.eval()
    crit = nn.CrossEntropyLoss(ignore_index=PAD)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    loss_sum, token_count = 0.0, 0
    all_preds = []
    
    for src, lens, tgt in loader:
        src, lens, tgt = src.to(DEVICE), lens.to(DEVICE), tgt.to(DEVICE)
        # 1. Dev Loss
        out = model(src, lens, tgt, teacher_forcing_ratio=0.0)
        n = int(tgt[:, 1:].ne(PAD).sum())
        loss = crit(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        loss_sum += float(loss) * n
        token_count += n
        
        # 2. Generation Dev F1
        gen_ids = model.generate(src, lens, max_target=MAX_TARGET)
        for seq in gen_ids.cpu().tolist():
            all_preds.append(decode_tokens(seq))
            
    dev_loss = loss_sum / token_count
    refs = [r["answer"] for r in raw_split_rows]
    metrics = compute_metrics(all_preds, refs)
    return dev_loss, metrics, all_preds

def train_model(model_name, model, train_ds, dev_ds, raw_dev_rows, epochs=MAX_EPOCHS):
    print(f"\n==========================================")
    print(f"Memulai Training {model_name}")
    print(f"==========================================")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    
    best_dev_f1 = -1.0
    best_dev_loss = float("inf")
    best_state = None
    history = []
    
    start_time = time.perf_counter()
    for ep in range(epochs):
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  generator=torch.Generator().manual_seed(SEED + ep))
        model.train()
        sum_loss, sum_tokens = 0.0, 0
        for src, lens, tgt in train_loader:
            src, lens, tgt = src.to(DEVICE), lens.to(DEVICE), tgt.to(DEVICE)
            optimizer.zero_grad()
            out = model(src, lens, tgt, teacher_forcing_ratio=0.5)
            loss = criterion(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            n = int(tgt[:, 1:].ne(PAD).sum())
            sum_loss += float(loss.detach()) * n
            sum_tokens += n
            
        train_loss = sum_loss / sum_tokens
        dev_loss, dev_met, _ = evaluate_dev(model, dev_ds, raw_dev_rows)
        
        ep_info = {
            "epoch": ep + 1,
            "train_loss": train_loss,
            "dev_loss": dev_loss,
            "dev_EM": dev_met["EM"],
            "dev_token_F1": dev_met["token_F1"],
            "dev_unique": dev_met["unique_predictions"],
            "dev_top_conc": dev_met["top_1_concentration"]
        }
        history.append(ep_info)
        
        is_best = (dev_met["token_F1"] > best_dev_f1) or (dev_met["token_F1"] == best_dev_f1 and dev_loss < best_dev_loss)
        if is_best:
            best_dev_f1 = dev_met["token_F1"]
            best_dev_loss = dev_loss
            best_state = copy.deepcopy(model.state_dict())
            marker = " [BEST SAVED]"
        else:
            marker = ""
            
        print(f"Epoch {ep+1:02d}/{epochs:02d} | Train Loss: {train_loss:.3f} | Dev Loss: {dev_loss:.3f} | Dev F1: {dev_met['token_F1']:.2f}% | EM: {dev_met['EM']:.1f}%{marker}")
        
    total_time = time.perf_counter() - start_time
    print(f"Training {model_name} Selesai dalam {total_time:.1f} detik ({total_time/60:.2f} menit).")
    
    # Muat state terbaik
    model.load_state_dict(best_state)
    return history, total_time

history_a, time_a = train_model("Varian A (Plain GRU)", model_a, train_dataset, dev_dataset, raw_data["dev"])
history_b, time_b = train_model("Varian B (GRU+Attention)", model_b, train_dataset, dev_dataset, raw_data["dev"])


Memulai Training Varian A (Plain GRU)


Epoch 01/10 | Train Loss: 4.988 | Dev Loss: 4.977 | Dev F1: 30.10% | EM: 20.0% [BEST SAVED]


Epoch 02/10 | Train Loss: 4.098 | Dev Loss: 4.995 | Dev F1: 30.29% | EM: 20.0% [BEST SAVED]


Epoch 03/10 | Train Loss: 3.564 | Dev Loss: 5.070 | Dev F1: 31.00% | EM: 20.0% [BEST SAVED]


Epoch 04/10 | Train Loss: 3.073 | Dev Loss: 5.133 | Dev F1: 31.90% | EM: 20.0% [BEST SAVED]


...

Epoch 06/10 | Train Loss: 2.286 | Dev Loss: 5.410 | Dev F1: 31.58% | EM: 20.0% [BEST SAVED]


Epoch 07/10 | Train Loss: 1.878 | Dev Loss: 5.730 | Dev F1: 31.27% | EM: 20.0%


Epoch 08/10 | Train Loss: 1.538 | Dev Loss: 5.895 | Dev F1: 31.28% | EM: 20.0%


Epoch 09/10 | Train Loss: 1.227 | Dev Loss: 6.213 | Dev F1: 31.40% | EM: 20.0%


Epoch 10/10 | Train Loss: 0.957 | Dev Loss: 6.458 | Dev F1: 31.15% | EM: 20.0%
Training Varian B (GRU+Attention) Selesai dalam 190.8 detik (3.18 menit).


## 8. Evaluasi Final pada Test Set (Karantina Dibuka)

Test set (500 sampel) **hanya dievaluasi tepat satu kali** menggunakan checkpoint terbaik dari Varian A dan Varian B.
Metrik yang dilaporkan mencakup:
- **EM (Exact Match)**
- **Token F1**
- **Keragaman Jawaban:** Jumlah prediksi unik dan konsentrasi jawaban teratas (mendeteksi *unigram collapse*)
- **Keberhasilan Non-Yes/No:** Jumlah jawaban EM yang bukan merupakan tebakan biner "Yes"/"No".

In [9]:
@torch.no_grad()
def evaluate_test(model, dataset, raw_split_rows):
    model.eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    all_preds = []
    for src, lens, _ in loader:
        src, lens = src.to(DEVICE), lens.to(DEVICE)
        gen_ids = model.generate(src, lens, max_target=MAX_TARGET)
        for seq in gen_ids.cpu().tolist():
            all_preds.append(decode_tokens(seq))
    refs = [r["answer"] for r in raw_split_rows]
    return compute_metrics(all_preds, refs), all_preds

test_metrics_a, test_preds_a = evaluate_test(model_a, test_dataset, raw_data["test"])
test_metrics_b, test_preds_b = evaluate_test(model_b, test_dataset, raw_data["test"])

print("=== HASIL EVALUASI TEST SET (500 SAMPEL) ===")
print(f"{'Metrik':<32} | {'Varian A (Plain GRU)':<22} | {'Varian B (GRU+Attention)':<24}")
print("-" * 84)
print(f"{'Exact Match (EM) (%)':<32} | {test_metrics_a['EM']:<22.2f} | {test_metrics_b['EM']:<24.2f}")
print(f"{'Token F1 (%)':<32} | {test_metrics_a['token_F1']:<22.2f} | {test_metrics_b['token_F1']:<24.2f}")
print(f"{'Jumlah Jawaban Unik':<32} | {test_metrics_a['unique_predictions']:<22d} | {test_metrics_b['unique_predictions']:<24d}")
print(f"{'Konsentrasi Jawaban Teratas (%)':<32} | {test_metrics_a['top_1_concentration']:<22.2f} | {test_metrics_b['top_1_concentration']:<24.2f}")
print(f"{'Prediksi Paling Sering':<32} | {test_metrics_a['top_1_prediction']:<22s} | {test_metrics_b['top_1_prediction']:<24s}")
print(f"{'Total Exact Match (contoh)':<32} | {test_metrics_a['total_em_count']:<22d} | {test_metrics_b['total_em_count']:<24d}")
print(f"{'  - EM Yes/No':<32} | {test_metrics_a['yes_no_em_count']:<22d} | {test_metrics_b['yes_no_em_count']:<24d}")
print(f"{'  - EM Non-Yes/No (Substantif)':<32} | {test_metrics_a['non_yes_no_em_count']:<22d} | {test_metrics_b['non_yes_no_em_count']:<24d}")

=== HASIL EVALUASI TEST SET (500 SAMPEL) ===
Metrik                           | Varian A (Plain GRU)   | Varian B (GRU+Attention)
------------------------------------------------------------------------------------
Exact Match (EM) (%)             | 20.00                  | 19.80                   
Token F1 (%)                     | 30.74                  | 32.06                   
Jumlah Jawaban Unik              | 122                    | 219                     
Konsentrasi Jawaban Teratas (%)  | 21.00                  | 20.80                   
Prediksi Paling Sering           | yes                    | yes                     
Total Exact Match (contoh)       | 100                    | 99                      
  - EM Yes/No                    | 100                    | 99                      
  - EM Non-Yes/No (Substantif)   | 0                      | 0                       


## 9. Analisis Kualitatif & Pola Kesalahan (*Error Analysis*)

Menampilkan:
1. 10 contoh pertama secara acak tanpa seleksi.
2. 5 contoh jawaban benar (Exact Match).
3. 5 contoh kegagalan beserta kategori pola kesalahannya.

In [10]:
print("=== 10 CONTOH PERTAMA TEST SET TANPA SELEKSI ===")
for i in range(10):
    gold = raw_data["test"][i]["answer"]
    q = raw_data["test"][i]["question"]
    p_a = test_preds_a[i] or "<kosong>"
    p_b = test_preds_b[i] or "<kosong>"
    print(f"\n[Test #{i+1}] Q: {q}")
    print(f"  Gold     : {gold}")
    print(f"  Varian A : {p_a}")
    print(f"  Varian B : {p_b}")

print("\n\n=== CONTOH TEREVALUASI BENAR (EXACT MATCH) PADA VARIAN B ===")
correct_samples = [i for i, (p, r) in enumerate(zip(test_preds_b, [r["answer"] for r in raw_data["test"]]))
                   if tokenize(p) == tokenize(r)]
print(f"Ditemukan {len(correct_samples)} contoh Exact Match pada Varian B:")
for i in correct_samples[:5]:
    q = raw_data["test"][i]["question"]
    gold = raw_data["test"][i]["answer"]
    p_b = test_preds_b[i]
    print(f"\n[Test #{i+1}] Q: {q}")
    print(f"  Jawaban Model: {p_b} (Gold: {gold})")

print("\n\n=== CONTOH KEGAGALAN DAN ANALISIS POLA KESALAHAN ===")
failed_samples = [i for i, (p, r) in enumerate(zip(test_preds_b, [r["answer"] for r in raw_data["test"]]))
                  if tokenize(p) != tokenize(r)]
for count, i in enumerate(failed_samples[:5]):
    q = raw_data["test"][i]["question"]
    gold = raw_data["test"][i]["answer"]
    p_b = test_preds_b[i] or "<kosong>"
    print(f"\n[Gagal #{count+1} - Test #{i+1}] Q: {q}")
    print(f"  Gold        : {gold}")
    print(f"  Prediksi B  : {p_b}")

=== 10 CONTOH PERTAMA TEST SET TANPA SELEKSI ===

[Test #1] Q: How does the code add a user in the given buckets default object access control list ?
  Gold     : as an owner
  Varian A : by
  Varian B : by the

[Test #2] Q: Does the code add a user in the given buckets default object access control list as an owner ?
  Gold     : Yes
  Varian A : yes
  Varian B : yes

[Test #3] Q: Where does the code add a user as an owner ?
  Gold     : in the given buckets default object access control list
  Varian A : in a
  Varian B : in the

[Test #4] Q: Does the code create a simple sdist tarball at dist_path ?
  Gold     : Yes
  Varian A : yes
  Varian B : yes

[Test #5] Q: How does the code create a simple sdist tarball ?
  Gold     : at dist_path
  Varian A : by the
  Varian B : with the given

[Test #6] Q: What does the code create at dist_path ?
  Gold     : a simple sdist tarball
  Varian A : a file
  Varian B : a file with the given file

[Test #7] Q: Where does le dirs set ?
  Gold     

## 10. Penyimpanan Checkpoint, Verifikasi Reload & Ekspor Hasil

Menyimpan model ke dalam disk, memuat kembali untuk menguji determinisme (*bit-for-bit check* via `torch.equal`), dan mengekspor seluruh prediksi ke format JSONL.

In [11]:
# 1. Simpan checkpoint
ckpt_a_path = OUTPUT_DIR / "variant_a.pt"
ckpt_b_path = OUTPUT_DIR / "variant_b.pt"

torch.save({"model_state": model_a.state_dict(), "vocab": vocab,
            "config": {"embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "vocab_size": len(vocab)}}, ckpt_a_path)
torch.save({"model_state": model_b.state_dict(), "vocab": vocab,
            "config": {"embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "vocab_size": len(vocab)}}, ckpt_b_path)

# 2. Uji Reload Determinism pada Varian B
reloaded_model_b = build_model_b().to(DEVICE)
state_b = torch.load(ckpt_b_path, map_location=DEVICE, weights_only=True)
reloaded_model_b.load_state_dict(state_b["model_state"])

test_probe_src = test_dataset.tensors[0][:10].to(DEVICE)
test_probe_len = test_dataset.tensors[1][:10].to(DEVICE)

orig_out = model_b.generate(test_probe_src, test_probe_len)
reloaded_out = reloaded_model_b.generate(test_probe_src, test_probe_len)

assert torch.equal(orig_out, reloaded_out), "Hasil checkpoint yang dimuat ulang tidak identik secara biner!"
print("Verifikasi Reload: CHECKPOINT DETERMINISTIK IDENTIK 100% LULUS.")

# 3. Ekspor Prediksi ke JSONL
def export_jsonl(path, rows, predictions):
    with open(path, "w", encoding="utf-8") as f:
        for r, p in zip(rows, predictions):
            f.write(json.dumps({**r, "prediction": p}, ensure_ascii=False) + "\n")

export_jsonl(OUTPUT_DIR / "predictions_variant_a.jsonl", raw_data["test"], test_preds_a)
export_jsonl(OUTPUT_DIR / "predictions_variant_b.jsonl", raw_data["test"], test_preds_b)

# 4. Ekspor Rangkuman Metrik Final
final_summary = {
    "hardware": {
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
    },
    "data": {
        "train_size": len(raw_data["train"]),
        "dev_size": len(raw_data["dev"]),
        "test_size": len(raw_data["test"]),
        "vocab_size": len(vocab),
        "max_source": MAX_SOURCE,
        "max_target": MAX_TARGET
    },
    "variant_a_plain": {
        "parameters": params_a,
        "training_time_sec": time_a,
        "test_metrics": test_metrics_a
    },
    "variant_b_attention": {
        "parameters": params_b,
        "training_time_sec": time_b,
        "test_metrics": test_metrics_b
    }
}

(OUTPUT_DIR / "final_results.json").write_text(json.dumps(final_summary, indent=2))
print(f"Seluruh berkas eksperimen berhasil disimpan di: {OUTPUT_DIR.resolve()}")
print("Selesai.")

Verifikasi Reload: CHECKPOINT DETERMINISTIK IDENTIK 100% LULUS.
Seluruh berkas eksperimen berhasil disimpan di: /home/inaka/Codes/LAB_IC/CodeQA/mini_attention_results
Selesai.
